# 00. セットアップ

各トピックを動かす前に、1回だけ実行します。

ここでやるのは **入れ物を作ること** だけです。

- カタログとスキーマ (テーブルを置く場所)
- Volume (ファイルを置く場所)

取り込むデータは各トピックのノートブックが自分で用意するので、ここでは作りません。

すべて `IF NOT EXISTS` なので、何度実行しても問題ありません。

## 準備

このノートブックはローカルのPythonで動きますが、処理自体はDatabricks側で実行されます。
その橋渡しをするのが databricks-connect で、`DatabricksSession` がその入口です。

In [ ]:
from databricks.connect import DatabricksSession

# serverless(True) = Databricks側のサーバーレスコンピュートに接続する
spark = DatabricksSession.builder.profile("free").serverless(True).getOrCreate()

In [ ]:
CATALOG = "tech_survey"

# メダリオンアーキテクチャの3層。生データ -> 整形済み -> 集計済み と段階を分ける考え方
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD = "gold"

# テーブル以外の置き場所。取り込み元ファイルとチェックポイントをここに置く
SCHEMA_OPS = "ops"

# Lakeflow Declarative Pipelines (08, 09) の出力先。他と混ざらないよう分けておく
SCHEMA_LDP = "ldp"

## カタログ・スキーマ・Volumeを作る

Unity Catalog では `カタログ.スキーマ.テーブル` の3階層で名前が決まります。
Volume はファイルを置くための領域で、`/Volumes/カタログ/スキーマ/Volume名` というパスでアクセスします。

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")

for schema in (SCHEMA_BRONZE, SCHEMA_SILVER, SCHEMA_GOLD, SCHEMA_OPS, SCHEMA_LDP):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")

# 取り込み元のファイルを置く場所
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA_OPS}.landing")

# ストリーム処理が「どこまで読んだか」を記録する場所
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA_OPS}.checkpoints")

## 確認

In [ ]:
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))

In [ ]:
display(spark.sql(f"SHOW VOLUMES IN {CATALOG}.{SCHEMA_OPS}"))

ここまで成功していれば `01_auto_loader` から順に進められます。

各ノートブックは `landing/<トピック名>/` に自分の取り込み元データを作るので、互いに干渉しません。

## 後片付け

最初からやり直したいときだけ実行します。
カタログ配下のテーブル・Volume・ファイルがすべて消えるので注意してください。

In [ ]:
# spark.sql(f"DROP CATALOG IF EXISTS {CATALOG} CASCADE")